# Bayesian Edge Detection

Original notebook by Raj Magesh Gauthaman (Fall 2025). Revised for Fall 2026: histogram estimates of the two distributions, the threshold from the prior and the loss, several filters at one pixel, and domain transfer.

Reference: S. Konishi, A. L. Yuille, J. M. Coughlan and S. C. Zhu, [Statistical edge detection: learning and evaluating edge cues](https://doi.org/10.1109/TPAMI.2003.1159946), IEEE TPAMI 25(1), 2003, and the notes for Lecture 4 (Bayesian Edge Detection).

If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Run the notebook from the homework folder, so that the folder `data` is next to it. Every figure and number that a question asks for must appear in your PDF: when a question asks you to vary a parameter, copy the relevant cell into the code cell below the question (or write a loop) instead of editing and re-running the cell above, which would overwrite the earlier figure.

---

This notebook implements the Bayesian edge detector of Lecture 4. A filter turns the image around pixel $x$ into one number, the filter response $f(I(x))$. From annotated images we learn the two class-conditional distributions $P(f \mid y=+1)$ (edge) and $P(f \mid y=-1)$ (non-edge), and at every pixel we decide with the log-likelihood ratio test

$$\log \frac{P(f \mid y=+1)}{P(f \mid y=-1)} > T, \qquad T = \log\frac{P(y=-1)}{P(y=+1)} + \log\frac{c_{FP}}{c_{FN}},$$

where $c_{FP}$ is the cost of a false positive and $c_{FN}$ the cost of a missed edge.

## Utilities

In [ ]:
from __future__ import annotations
import os

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from PIL import Image

%matplotlib inline

DATA = "data/edge_detection"
if not os.path.isdir(DATA):
    raise FileNotFoundError(f"The folder {DATA} was not found. Start Jupyter in the homework folder, "
                            "so that the folder data is next to this notebook.")
TRAIN_IDS = (0, 1, 2, 3, 4)   # the distributions are learned on these images
TEST_IDS = (5, 6)             # and evaluated on these


def load_data(id: int = 0) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.bool_]]:
    """Load image `id` (0 to 6) as grey levels in [0, 1], and its hand-annotated edge map."""
    image = np.array(Image.open(f"{DATA}/images/{id}.jpg").convert("L")).astype(float) / 255
    edge_map = np.array(Image.open(f"{DATA}/edge_maps/{id}.bmp").convert("1"))
    return image, edge_map


def load_colour(id: int = 0) -> npt.NDArray[np.float64]:
    """Load image `id` (0 to 6) as an RGB array in [0, 1]."""
    return np.array(Image.open(f"{DATA}/images/{id}.jpg").convert("RGB")).astype(float) / 255


def edge_labels(edge_map: npt.NDArray[np.bool_], tolerance: int = 1) -> npt.NDArray[np.bool_]:
    """Labels y(x): True (y = +1) for pixels within `tolerance` pixels of an annotated boundary.

    The annotated boundaries are one pixel wide and can be off by a pixel, so the pixels
    next to a boundary are counted as edge pixels too.
    """
    if tolerance == 0:
        return edge_map.copy()
    size = 2 * tolerance + 1
    return ndi.binary_dilation(edge_map, structure=np.ones((size, size), dtype=bool))


def filter_response(image: npt.NDArray[np.float64], kind: str = "gradient", sigma: float = 1.0) -> npt.NDArray[np.float64]:
    """Filter response f(I(x)) at every pixel.

    kind = "gradient":  |grad(G_sigma * I)|, the gradient magnitude after Gaussian smoothing
    kind = "laplacian": |sigma^2 Laplacian(G_sigma * I)|, the magnitude of the Laplacian of Gaussian
    kind = "colour":    gradient magnitude of the two colour-opponent channels (red-green and
                        yellow-blue); `image` must then be an RGB array
    """
    if kind == "gradient":
        return ndi.gaussian_gradient_magnitude(image, sigma)
    if kind == "laplacian":
        return np.abs(ndi.gaussian_laplace(image, sigma)) * sigma**2
    if kind == "colour":
        r, g, b = image[..., 0], image[..., 1], image[..., 2]
        red_green = (r - g) / np.sqrt(2)
        yellow_blue = (r + g - 2 * b) / np.sqrt(6)
        return np.hypot(ndi.gaussian_gradient_magnitude(red_green, sigma), ndi.gaussian_gradient_magnitude(yellow_blue, sigma))
    raise ValueError(f"unknown filter {kind}")


def learn_distributions(responses, labels, n_bins: int = 64, max_value: float | None = None):
    """Histogram estimates of P(f | y=+1) and P(f | y=-1).

    Returns the bin edges and the two histograms, each normalised to sum to one.
    The last bin is open-ended: responses above `max_value` fall into it.
    """
    f, y = np.ravel(responses), np.ravel(labels)
    if max_value is None:
        max_value = np.quantile(f, 0.999)
    bins = np.linspace(0, max_value, n_bins + 1)
    bins[-1] = np.inf
    p_on = np.histogram(f[y], bins=bins)[0].astype(float)
    p_off = np.histogram(f[~y], bins=bins)[0].astype(float)
    return bins, p_on / p_on.sum(), p_off / p_off.sum()


def log_likelihood_ratio(responses, bins, p_on, p_off, floor: float = 1e-6):
    """log P(f | y=+1) / P(f | y=-1), read off the two histograms.

    The probabilities are floored so that an empty bin does not give plus or minus infinity.
    """
    k = np.clip(np.digitize(responses, bins) - 1, 0, len(p_on) - 1)
    return np.log((p_on[k] + floor) / (p_off[k] + floor))


def decision_threshold(prior_edge: float, c_fp: float = 1.0, c_fn: float = 1.0) -> float:
    """T = log P(y=-1)/P(y=+1) + log c_FP/c_FN."""
    return float(np.log((1 - prior_edge) / prior_edge) + np.log(c_fp / c_fn))


def risk(decisions, labels, c_fp: float = 1.0, c_fn: float = 1.0) -> float:
    """Average loss per pixel: c_FN for every missed edge pixel, c_FP for every false positive."""
    d, y = np.ravel(decisions), np.ravel(labels)
    return float((c_fn * np.sum(~d & y) + c_fp * np.sum(d & ~y)) / y.size)


def compute_roc(scores, labels):
    """ROC curve of the test `score > threshold` as the threshold is swept.

    Returns the false positive rates, the true positive (hit) rates and the area under the curve.
    """
    s, y = np.ravel(scores), np.ravel(labels)
    order = np.argsort(-s, kind="stable")
    s, y = s[order], y[order]
    tp, fp = np.cumsum(y), np.cumsum(~y)
    last = np.r_[np.nonzero(np.diff(s))[0], len(s) - 1]   # one point per distinct score
    tpr = np.r_[0.0, tp[last] / y.sum()]
    fpr = np.r_[0.0, fp[last] / (~y).sum()]
    area = float(np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2))
    return fpr, tpr, area


def bin_centres(bins):
    """Centres of the histogram bins (the open-ended last bin is drawn with the width of the others)."""
    edges = bins.copy()
    edges[-1] = edges[-2] + (edges[1] - edges[0])
    return (edges[:-1] + edges[1:]) / 2


def show(images, titles, cmap="gray", **kwargs):
    """Plot a row of images."""
    fig, axes = plt.subplots(1, len(images), figsize=(3.6 * len(images), 3))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(image, cmap=cmap, **kwargs)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## The data

Seven photographs from the Berkeley Segmentation Dataset with hand-annotated object boundaries (the same images as in the 2025 homework). The distributions are learned on images 0 to 4 and tested on images 5 and 6.

A pixel is labelled edge ($y=+1$) if it lies within one pixel of an annotated boundary, and non-edge ($y=-1$) otherwise. The annotations follow the convention described in Lecture 4: an edge is the boundary of an object (or a significant change of texture, such as lettering), and strong intensity variation inside a region (fur, grass, water) belongs to the non-edge class.

In [ ]:
images, colour_images, labels = {}, {}, {}
for i in range(7):
    images[i], edge_map = load_data(i)
    colour_images[i] = load_colour(i)
    labels[i] = edge_labels(edge_map)


def stack(ids, kind: str = "gradient", sigma: float = 1.0):
    """Concatenate the filter responses and the labels of several images into two 1-D arrays."""
    source = colour_images if kind == "colour" else images
    f = np.concatenate([filter_response(source[i], kind, sigma).ravel() for i in ids])
    y = np.concatenate([labels[i].ravel() for i in ids])
    return f, y


show([images[2], labels[2]], ["image 2", "edge pixels (y = +1)"])
for i in range(7):
    print(f"image {i}: {100 * labels[i].mean():.1f}% of the pixels are edge pixels")

## Filter responses

The derivative is computed after Gaussian smoothing, because differentiation amplifies noise. The scale $\sigma$ of the Gaussian selects the size of structure the filter responds to. The Laplacian of Gaussian is shown for comparison.

In [ ]:
show(
    [filter_response(images[2], "gradient", 1.0), filter_response(images[2], "gradient", 4.0), filter_response(images[2], "laplacian", 2.0)],
    [r"$|\nabla(G_\sigma * I)|$, $\sigma=1$", r"$|\nabla(G_\sigma * I)|$, $\sigma=4$", r"$|\sigma^2 \nabla^2(G_\sigma * I)|$, $\sigma=2$"],
)

## Learning the two distributions

Apply the filter to every training pixel, split the responses into two pools according to the label, and build a histogram of each pool. Each histogram is normalised by its own total, so it estimates a distribution over responses conditioned on the label. The last bin collects every response above roughly the 99.9th percentile, which is why the curves jump up at the right end.

In [ ]:
sigma = 1.0   # change this for Question 8.1

f_train, y_train = stack(TRAIN_IDS, "gradient", sigma)
bins, p_on, p_off = learn_distributions(f_train, y_train)
prior_edge = y_train.mean()
print(f"P(y=+1) on the training images: {prior_edge:.4f}")

centres = bin_centres(bins)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, log_scale in zip(axes, (False, True)):
    ax.step(centres, p_on, where="mid", color="tab:red", label=r"edge, $P(f \mid y=+1)$")
    ax.step(centres, p_off, where="mid", color="tab:blue", label=r"non-edge, $P(f \mid y=-1)$")
    ax.set_xlabel(rf"filter response $f = |\nabla(G_\sigma * I)|$, $\sigma = {sigma:g}$")
    ax.set_ylabel("probability per bin")
    if log_scale:
        ax.set_yscale("log")
        ax.set_title("logarithmic vertical axis")
    else:
        ax.set_title("linear vertical axis")
    ax.legend()
fig.tight_layout()
plt.show()

### Question 8.1 (3 points)

- Plot the two class-conditional distributions of the gradient magnitude for $\sigma = 1$ and for $\sigma = 2$ (copy the cell above into the code cell below and run it for both values), and report the fraction of training pixels labelled edge, $P(y=+1)$. **(1 point)**
- Why does the non-edge distribution pile up near zero, and why do the two distributions overlap? What does the overlap imply for the Bayes risk, and how does the overlap change from $\sigma = 1$ to $\sigma = 2$? **(2 points)**

In [ ]:
# Your code for Question 8.1

**Your answers for Question 8.1:**

## From the likelihood ratio to a decision

The log-likelihood ratio is computed at every pixel of a test image from the two learned histograms and compared with the threshold $T$. The costs below say that missing an edge is five times worse than a false positive.

In [ ]:
sigma = 1.0
f_train, y_train = stack(TRAIN_IDS, "gradient", sigma)
bins, p_on, p_off = learn_distributions(f_train, y_train)
prior_edge = y_train.mean()

c_fp, c_fn = 1.0, 5.0
T = decision_threshold(prior_edge, c_fp, c_fn)
print(f"T = log P(y=-1)/P(y=+1) + log c_FP/c_FN = {np.log((1 - prior_edge) / prior_edge):.3f} + {np.log(c_fp / c_fn):.3f} = {T:.3f}")

test_id = 5
llr = log_likelihood_ratio(filter_response(images[test_id], "gradient", sigma), bins, p_on, p_off)
show(
    [images[test_id], llr, llr > T, llr > 0, labels[test_id]],
    [f"test image {test_id}", "log-likelihood ratio", f"decision at T = {T:.2f}", "decision at T = 0", "ground truth"],
)
for name, d in (("T", llr > T), ("0", llr > 0)):
    print(f"test image {test_id}: {100 * d.mean():.1f}% of the pixels are declared edges at threshold {name}")

# The learned log-likelihood ratio as a function of the filter response
plt.figure(figsize=(5.5, 3.2))
plt.plot(bin_centres(bins), log_likelihood_ratio(bin_centres(bins), bins, p_on, p_off), marker=".", color="k")
plt.axhline(T, color="tab:orange", ls="--", label=f"T = {T:.2f}")
plt.axhline(0, color="0.6", lw=1)
plt.xlabel(r"filter response $f = |\nabla(G_\sigma * I)|$, $\sigma = 1$")
plt.ylabel("log-likelihood ratio")
plt.legend()
plt.show()

In [ ]:
# Empirical risk on the test images as a function of the threshold
f_test, y_test = stack(TEST_IDS, "gradient", sigma)
llr_test = log_likelihood_ratio(f_test, bins, p_on, p_off)

thresholds = np.linspace(-3, 6, 181)
risks = np.array([risk(llr_test > t, y_test, c_fp, c_fn) for t in thresholds])
print(f"P(y=+1) on the test images: {y_test.mean():.4f}")
print(f"risk at the Bayes threshold T = {T:.2f}: {risk(llr_test > T, y_test, c_fp, c_fn):.4f}")
print(f"lowest empirical risk: {risks.min():.4f} at threshold {thresholds[risks.argmin()]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(thresholds, risks, color="k")
axes[0].axvline(T, color="tab:orange", ls="--", label=f"Bayes threshold T = {T:.2f}")
axes[0].set_xlabel("threshold on the log-likelihood ratio")
axes[0].set_ylabel("average loss per pixel on the test images")
axes[0].legend()

# ROC curves: thresholding the log-likelihood ratio, and thresholding the raw response
for scores, name, color, style in ((llr_test, "log-likelihood ratio", "tab:purple", "-"), (f_test, "raw response f", "tab:green", "--")):
    fpr, tpr, area = compute_roc(scores, y_test)
    axes[1].plot(fpr, tpr, color=color, ls=style, label=f"{name} (area {area:.3f})")
axes[1].plot([0, 1], [0, 1], color="0.7", lw=1)
axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("hit rate")
axes[1].set_title("ROC curves on the test images")
axes[1].legend()
fig.tight_layout()
plt.show()

### Question 8.2 (4 points)

- With $c_{FP}=1$ and $c_{FN}=5$, report the threshold $T$ and show the decision maps on test image 5 at this $T$ and at $T=0$. Describe the difference between the two maps. **(1 point)**
- Plot the empirical risk on the test images as a function of the threshold. Is its minimum close to the threshold $T$ predicted by Bayes decision theory? Give one reason why the two need not coincide exactly. **(2 points)**
- The two ROC curves come from thresholding the log-likelihood ratio and from thresholding the raw response $f$. Why are they almost identical here (look at the plot of the log-likelihood ratio against $f$)? When would the two tests give different ROC curves? **(1 point)**

In [ ]:
# Your code for Question 8.2

**Your answers for Question 8.2:**

## Many filters at one pixel

One filter is often ambiguous. With a bank of filters $f_1, \dots, f_M$ at the same pixel, the joint distribution $P(f_1, \dots, f_M \mid y)$ is hard to learn, so we assume the responses are conditionally independent given the label. The log-likelihood ratio is then a sum of one term per filter:

$$\log\frac{P(f_1,\dots,f_M \mid y=+1)}{P(f_1,\dots,f_M \mid y=-1)} = \sum_{i=1}^{M}\log\frac{P(f_i \mid y=+1)}{P(f_i \mid y=-1)}.$$

We use five cues: the gradient magnitude at three scales, the Laplacian of Gaussian, and a colour cue (the gradient of the two colour-opponent channels), as in Konishi et al.

In [ ]:
cues = {
    "gradient, sigma = 1": ("gradient", 1.0),
    "gradient, sigma = 2": ("gradient", 2.0),
    "gradient, sigma = 4": ("gradient", 4.0),
    "Laplacian, sigma = 2": ("laplacian", 2.0),
    "colour, sigma = 2": ("colour", 2.0),
}

llr_per_cue, train_responses = {}, {}
for name, (kind, s) in cues.items():
    f_tr, y_tr = stack(TRAIN_IDS, kind, s)
    f_te, y_te = stack(TEST_IDS, kind, s)
    b, pon, poff = learn_distributions(f_tr, y_tr)
    llr_per_cue[name] = log_likelihood_ratio(f_te, b, pon, poff)
    train_responses[name] = f_tr

combination = sum(llr_per_cue.values())   # the independence assumption

fig, ax = plt.subplots(figsize=(5.5, 5))
for name, scores in list(llr_per_cue.items()) + [("sum of all five", combination)]:
    fpr, tpr, area = compute_roc(scores, y_te)
    ax.plot(fpr, tpr, lw=2.5 if name == "sum of all five" else 1.2, label=f"{name} (area {area:.3f})")
ax.plot([0, 1], [0, 1], color="0.7", lw=1)
ax.set_xlabel("false positive rate")
ax.set_ylabel("hit rate")
ax.set_title("ROC curves on the test images")
ax.legend(fontsize=8)
plt.show()

# Add each cue in turn to the sigma = 2 gradient
base = "gradient, sigma = 2"
print(f"{'cue added to the sigma = 2 gradient':38s} {'ROC area':>9s} {'correlation at edge pixels':>28s}")
print(f"{'(none: the sigma = 2 gradient alone)':38s} {compute_roc(llr_per_cue[base], y_te)[2]:9.3f}")
for name in cues:
    if name == base:
        continue
    area = compute_roc(llr_per_cue[base] + llr_per_cue[name], y_te)[2]
    r = np.corrcoef(train_responses[base][y_tr], train_responses[name][y_tr])[0, 1]
    print(f"{name:38s} {area:9.3f} {r:28.2f}")

# Decisions at the Bayes threshold T (c_FP = 1, c_FN = 5) for one cue and for the sum of five
T = decision_threshold(y_tr.mean(), 1.0, 5.0)
for name, scores in ((base, llr_per_cue[base]), ("sum of all five", combination)):
    d = scores > T
    print(f"{name:16s}: {100 * d.mean():.1f}% of the test pixels declared edges, risk {risk(d, y_te, 1.0, 5.0):.4f}")

In [ ]:
# Are the cues independent given the label? Fine against coarse scale.
f1, y_tr = stack(TRAIN_IDS, "gradient", 1.0)
f4, _ = stack(TRAIN_IDS, "gradient", 4.0)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mask, title in ((axes[0], y_tr, "edge pixels (y = +1)"), (axes[1], ~y_tr, "non-edge pixels (y = -1)")):
    idx = rng.choice(np.nonzero(mask)[0], size=4000, replace=False)
    ax.scatter(f1[idx], f4[idx], s=2, alpha=0.4)
    r = np.corrcoef(f1[mask], f4[mask])[0, 1]
    ax.set_title(f"{title}: correlation {r:.2f}")
    ax.set_xlabel(r"$|\nabla(G_\sigma * I)|$, $\sigma=1$")
    ax.set_ylabel(r"$|\nabla(G_\sigma * I)|$, $\sigma=4$")
fig.tight_layout()
plt.show()

### Question 8.3 (4 points)

- Report the ROC area of each single cue and of the sum of all five log-likelihood ratios. Does combining the cues help? **(1 point)**
- The table adds each cue in turn to the $\sigma=2$ gradient (the first row is the $\sigma=2$ gradient alone). Which cue helps most? Why do the other two gradient scales add little or nothing? Use the correlations in the table. **(1 point)**
- Look at the scatter plots: are the cues conditionally independent given the label? At the same Bayes threshold $T$, the sum of five log-likelihood ratios declares many more pixels to be edges than the $\sigma=2$ gradient alone. Explain why, using what the independence assumption does with correlated cues. What does Lecture 4 suggest instead (see the Lecture 4 notes, pp. 39–41)? **(2 points)**

In [ ]:
# Your code for Question 8.3

**Your answers for Question 8.3:**

## Domain transfer

The statistics of images differ between domains. To study this under control we generate two synthetic domains with the same kind of boundaries: a *quiet* domain, where every region has a constant grey level, and a *textured* domain, where fine texture is added inside every region. The detector is trained on the quiet domain, where labels are available, and tested on the textured domain.

Three detectors are compared on the textured domain:

1. **source only**: both distributions learned on the quiet domain;
2. **transfer**: $P(f \mid y=+1)$ from the quiet domain, and $P(f \mid y=-1)$ re-estimated from *all* pixels of the textured images, without using their labels (most pixels are non-edge pixels);
3. **oracle**: both distributions learned from the labelled textured images (the same images it is evaluated on, so it is an in-sample reference).

All three use the same threshold $T$, computed from the prior of the quiet domain and the costs above.

In [ ]:
def make_scene(rng, size: int = 128, n_shapes: int = 5, texture: float = 0.0, noise: float = 0.02):
    """A synthetic image of ellipses and rectangles on a background, and its boundary map.

    Every region has a constant grey level. With texture > 0, fine-grained texture of roughly that
    amplitude is added inside every region; the boundaries and the region means do not change.
    """
    yy, xx = np.mgrid[0:size, 0:size]
    region = np.zeros((size, size), dtype=int)
    for k in range(1, n_shapes + 1):
        cy, cx = rng.uniform(0.15, 0.85, 2) * size
        a, b = rng.uniform(0.08, 0.25, 2) * size
        th = rng.uniform(0, np.pi)
        u = ((xx - cx) * np.cos(th) + (yy - cy) * np.sin(th)) / a
        v = (-(xx - cx) * np.sin(th) + (yy - cy) * np.cos(th)) / b
        inside = (u**2 + v**2 < 1) if rng.random() < 0.5 else ((np.abs(u) < 1) & (np.abs(v) < 1))
        region[inside] = k
    levels = rng.permutation(np.linspace(0.12, 0.88, n_shapes + 1)) + rng.uniform(-0.03, 0.03, n_shapes + 1)
    image = levels[region]
    if texture > 0:
        for k in range(n_shapes + 1):
            t = ndi.gaussian_filter(rng.normal(size=(size, size)), 0.8)
            image = image + (region == k) * texture * rng.uniform(0.6, 1.4) * t / t.std()
    image = ndi.gaussian_filter(image, 1.0) + rng.normal(scale=noise, size=(size, size))
    boundary = np.zeros((size, size), dtype=bool)
    boundary[:, 1:] |= region[:, 1:] != region[:, :-1]
    boundary[1:, :] |= region[1:, :] != region[:-1, :]
    return np.clip(image, 0, 1), boundary


rng = np.random.default_rng(2026)
quiet = [make_scene(rng, texture=0.0) for _ in range(20)]
textured = [make_scene(rng, texture=0.10) for _ in range(20)]
show([quiet[0][0], quiet[0][1], textured[0][0], textured[0][1]], ["quiet domain", "its boundaries", "textured domain", "its boundaries"])

In [ ]:
def stack_scenes(scenes, sigma: float = 1.0):
    f = np.concatenate([filter_response(im, "gradient", sigma).ravel() for im, _ in scenes])
    y = np.concatenate([edge_labels(b).ravel() for _, b in scenes])
    return f, y


sigma = 1.0
f_src, y_src = stack_scenes(quiet, sigma)
f_tgt, y_tgt = stack_scenes(textured, sigma)
max_value = np.quantile(np.r_[f_src, f_tgt], 0.999)      # the same bins for every histogram

bins, pon_src, poff_src = learn_distributions(f_src, y_src, max_value=max_value)
_, pon_tgt, poff_tgt = learn_distributions(f_tgt, y_tgt, max_value=max_value)
p_all_tgt = np.histogram(f_tgt, bins=bins)[0].astype(float)
p_all_tgt /= p_all_tgt.sum()                              # all target pixels, labels not used

T = decision_threshold(y_src.mean(), c_fp=1.0, c_fn=5.0)
detectors = {
    "source only": (pon_src, poff_src),
    "transfer": (pon_src, p_all_tgt),
    "oracle": (pon_tgt, poff_tgt),
}

centres = bin_centres(bins)
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.step(centres, pon_src, where="mid", color="tab:red", ls="--", label="edge, quiet domain")
ax.step(centres, pon_tgt, where="mid", color="tab:red", label="edge, textured domain")
ax.step(centres, poff_src, where="mid", color="tab:blue", ls="--", label="non-edge, quiet domain")
ax.step(centres, poff_tgt, where="mid", color="tab:blue", label="non-edge, textured domain")
ax.set_yscale("log")
ax.set_xlabel(r"$|\nabla(G_\sigma * I)|$, $\sigma = 1$")
ax.set_ylabel("probability per bin")
ax.legend(fontsize=8)
plt.show()

print(f"threshold T = {T:.2f}")
print(f"{'detector':12s} {'risk':>7s} {'false positive rate':>20s} {'hit rate':>9s}")
decision_maps = {}
for name, (pon, poff) in detectors.items():
    d = log_likelihood_ratio(f_tgt, bins, pon, poff) > T
    fpr = np.sum(d & ~y_tgt) / np.sum(~y_tgt)
    hit = np.sum(d & y_tgt) / np.sum(y_tgt)
    print(f"{name:12s} {risk(d, y_tgt, 1.0, 5.0):7.4f} {fpr:20.3f} {hit:9.3f}")
    image0 = filter_response(textured[0][0], "gradient", sigma)
    decision_maps[name] = log_likelihood_ratio(image0, bins, pon, poff) > T

show([textured[0][0]] + list(decision_maps.values()), ["textured image"] + [f"decisions: {n}" for n in decision_maps])

### Question 8.4 (3 points)

- Report the risk, the false positive rate and the hit rate of the three detectors on the textured domain, and show their decision maps. **(1 point)**
- Which of the two class-conditional distributions changes most between the quiet and the textured domain (use the histogram plot)? Explain why the source-only detector fails on the textured domain, and why the transfer detector recovers most of the oracle's performance without using the labels of the textured images. **(2 points)**

In [ ]:
# Your code for Question 8.4

**Your answers for Question 8.4:**